# ClearTrace — Feature Audit and v2 Redesign

This notebook audits `features_core_v1.csv` and creates the leakage-resistant `features_core_v2.csv` used for forecasting.

The audit focuses on:

1. Separating original pollutant observations from interpolation and proxy estimates.
2. Rebuilding pollutant lag and rolling features from original observations only.
3. Replacing future-aware station-availability summaries with causal missingness flags.
4. Validating neighbouring-station features chronologically.
5. Exporting an explicit and validated v2 feature contract.

`features_core_v1.csv` is retained as the historical baseline. `features_core_v2.csv` is the authoritative modelling dataset.

In [7]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "features_core_v1.csv"
)



df = pd.read_csv(DATA_PATH)
pollutants= ["pm25", "pm10", "no2", "co", "so2", "o3"]

interpolation_columns = [
    f"{pollutant}_interpolated"
    for pollutant in pollutants
]

print(interpolation_columns)

    


['pm25_interpolated', 'pm10_interpolated', 'no2_interpolated', 'co_interpolated', 'so2_interpolated', 'o3_interpolated']


In [9]:
interpolation_flags = df[interpolation_columns]

rows_with_interpolation = interpolation_flags.any(axis=1)

interpolation_count = rows_with_interpolation.sum()
interpolation_percentage = rows_with_interpolation.mean()*100

print("Rows affected:", interpolation_count)
print("Percentage:", round(interpolation_percentage, 2))

Rows affected: 15267
Percentage: 19.52


In [10]:
proxy_columns = [
    f"{pollutant}_proxy_imputed"
    for pollutant in pollutants]

proxy_flags = df[proxy_columns]
rows_with_proxy = proxy_flags.any(axis=1)

proxy_count = rows_with_proxy.sum()
proxy_percentage = rows_with_proxy.mean() * 100

print("Rows affected:", proxy_count)
print("Percentage:", round(proxy_percentage, 2))


Rows affected: 20621
Percentage: 26.37


In [11]:
so2_observation_counts = (
    df.groupby("station_name")["has_so2"]
    .sum()
    .sort_values()
)

so2_observation_counts.head(10)

station_name
Aya Nagar, New Delhi - IMD                   0
Burari Crossing, New Delhi - IMD             0
CRRI Mathura Road, New Delhi - IMD           0
IGI Airport (T3), Delhi - IMD                0
New Moti Bagh, Delhi - MHUA                  0
North Campus, DU, Delhi - IMD                0
Lodhi Road, New Delhi - IMD                  0
Chandni Chowk, Delhi - IITM                271
IHBAS, Dilshad Garden,New Delhi - CPCB    1332
R K Puram, Delhi - DPCC                   1454
Name: has_so2, dtype: int64

## Audit finding 1 — Structural proxy contamination

The v1 dataset contains neighbouring-station proxy estimates inside the main pollutant columns. At several stations, a pollutant was never originally observed, yet proxy estimates made it appear as though the station had a local pollutant history.

For example, seven stations never reported SO2 but received SO2 proxy estimates. Similar structural cases also exist for PM10 and O3.

The proxy estimates are not inherently “fake,” but they must not be represented as original local observations. The correction is therefore applied generically to all six pollutants using the original `has_<pollutant>` availability flags.

In [13]:
stations_without_so2 = so2_observation_counts[
    so2_observation_counts == 0
].index

print(stations_without_so2)

Index(['Aya Nagar, New Delhi - IMD', 'Burari Crossing, New Delhi - IMD',
       'CRRI Mathura Road, New Delhi - IMD', 'IGI Airport (T3), Delhi - IMD',
       'New Moti Bagh, Delhi - MHUA', 'North Campus, DU, Delhi - IMD',
       'Lodhi Road, New Delhi - IMD'],
      dtype='object', name='station_name')


In [14]:
rows_without_real_so2 = df[
    df["station_name"].isin(stations_without_so2)
]

proxy_so2_count = rows_without_real_so2[
    "so2_proxy_imputed"
].sum()

print("SO2 proxy values at stations with no real SO2:", proxy_so2_count)

SO2 proxy values at stations with no real SO2: 14390


In [15]:
for pollutant in pollutants:
    observation_column = f"has_{pollutant}"
    proxy_column = f"{pollutant}_proxy_imputed"

    observation_counts = (
        df.groupby("station_name")[observation_column]
        .sum()
    )

    stations_without_observations = observation_counts[
        observation_counts == 0
    ].index

    affected_rows = df[
        df["station_name"].isin(stations_without_observations)
    ]

    proxy_count = affected_rows[proxy_column].sum()

    print(
        pollutant,
        "stations without observations:",
        len(stations_without_observations),
        "proxy values:",
        proxy_count
    )

pm25 stations without observations: 0 proxy values: 0
pm10 stations without observations: 1 proxy values: 2058
no2 stations without observations: 0 proxy values: 0
co stations without observations: 0 proxy values: 0
so2 stations without observations: 7 proxy values: 14390
o3 stations without observations: 1 proxy values: 2058


In [17]:
df_v2 = df.copy()

In [18]:
df_v2["so2_observed"] = df_v2["so2"].where(
    df_v2["has_so2"]
)

In [19]:
print(
    "Observed SO2 values:",
    df_v2["so2_observed"].notna().sum()
)

print(
    "Original availability flags:",
    df_v2["has_so2"].sum()
)

Observed SO2 values: 50450
Original availability flags: 50450


In [20]:
aya_nagar = df_v2[
    df_v2["station_name"] == "Aya Nagar, New Delhi - IMD"
]

aya_nagar[
    [
        "timestamp_hour",
        "so2",
        "has_so2",
        "so2_proxy_imputed",
        "so2_observed",
    ]
].head()

,timestamp_hour,so2,has_so2,so2_proxy_imputed,so2_observed
6174,2026-04-10 21:00:00,12.100,False,True,NaN
6175,2026-04-10 22:00:00,12.500,False,True,NaN
6176,2026-04-10 23:00:00,13.700,False,True,NaN
6177,2026-04-11 00:00:00,12.675,False,True,NaN
6178,2026-04-11 01:00:00,11.650,False,True,NaN


In [21]:
for pollutant in pollutants:
    observed_column = f"{pollutant}_observed"

    df_v2[observed_column] = df_v2[pollutant].where(
        df_v2[f"has_{pollutant}"]
    )

In [22]:
for pollutant in pollutants:
    observed_count = df_v2[f"{pollutant}_observed"].notna().sum()
    availability_count = df_v2[f"has_{pollutant}"].sum()

    print(pollutant, observed_count, availability_count)

pm25 62828 62828
pm10 62334 62334
no2 62919 62919
co 61786 61786
so2 50450 50450
o3 61704 61704


In [23]:
for pollutant in pollutants:
    station_observed_column = f"station_observed_{pollutant}"

    df_v2[station_observed_column] = (
        df_v2.groupby("location_id")[f"has_{pollutant}"]
        .transform("any")
    )

In [25]:
for pollutant in pollutants:
    station_observed_column = f"station_observed_{pollutant}"

    station_availability = (
        df_v2.groupby("location_id")[station_observed_column]
        .first()
    )

    unavailable_stations = (
        ~station_availability
    ).sum()

    print(
        pollutant,
        "stations without observations:",
        unavailable_stations
    )

pm25 stations without observations: 0
pm10 stations without observations: 1
no2 stations without observations: 0
co stations without observations: 0
so2 stations without observations: 7
o3 stations without observations: 1


## Diagnostic separation of observations and estimates

Original observations are reconstructed as `<pollutant>_observed` using the corresponding `has_<pollutant>` flag.

Legacy proxy estimates and full-period station-availability summaries are temporarily separated into diagnostic columns so the v1 problem can be measured. These diagnostic columns are not used in the final model contract because full-period station summaries are unavailable causally during live inference.

In [26]:
for pollutant in pollutants:
    has_column = f"has_{pollutant}"
    station_observed_column = f"station_observed_{pollutant}"
    proxy_flag_column = f"{pollutant}_proxy_imputed"

    df_v2[f"{pollutant}_neighbor_estimate_v1"] = (
        df_v2[pollutant].where(
            df_v2[proxy_flag_column]
        )
    )

    df_v2[f"{pollutant}_temporary_missing"] = (
        df_v2[station_observed_column]
        & ~df_v2[has_column]
    )

    df_v2[f"{pollutant}_structural_missing"] = (
        ~df_v2[station_observed_column]
    )

In [27]:
for pollutant in pollutants:
    overlap = (
        df_v2[f"{pollutant}_observed"].notna()
        & df_v2[f"{pollutant}_neighbor_estimate_v1"].notna()
    ).sum()

    proxy_count = (
        df_v2[f"{pollutant}_neighbor_estimate_v1"]
        .notna()
        .sum()
    )

    original_proxy_count = (
        df_v2[f"{pollutant}_proxy_imputed"]
        .sum()
    )

    print(
        pollutant,
        "overlap:",
        overlap,
        "separated proxy values:",
        proxy_count,
        "original proxy flags:",
        original_proxy_count,
    )

pm25 overlap: 0 separated proxy values: 4792 original proxy flags: 4792
pm10 overlap: 0 separated proxy values: 6016 original proxy flags: 6016
no2 overlap: 0 separated proxy values: 4710 original proxy flags: 4710
co overlap: 0 separated proxy values: 5031 original proxy flags: 5031
so2 overlap: 0 separated proxy values: 18052 original proxy flags: 18052
o3 overlap: 0 separated proxy values: 6061 original proxy flags: 6061


In [28]:
aya_nagar_v2 = df_v2[
    df_v2["station_name"] == "Aya Nagar, New Delhi - IMD"
]

aya_nagar_v2[
    [
        "so2",
        "has_so2",
        "so2_observed",
        "so2_neighbor_estimate_v1",
        "so2_temporary_missing",
        "so2_structural_missing",
    ]
].head()

,so2,has_so2,so2_observed,so2_neighbor_estimate_v1,so2_temporary_missing,so2_structural_missing
6174,12.100,False,NaN,12.100,False,True
6175,12.500,False,NaN,12.500,False,True
6176,13.700,False,NaN,13.700,False,True
6177,12.675,False,NaN,12.675,False,True
6178,11.650,False,NaN,11.650,False,True


## Diagnostic columns created during the audit

The temporary `*_neighbor_estimate_v1`, `*_temporary_missing` and `*_structural_missing` columns expose how v1 mixed local observations with neighbouring-station estimates.

These columns do not measure whether a proxy value is “reliable,” and they are not final model features. They are excluded from the v2 model contract and replaced by observed-only temporal features, causal missingness flags and separately validated spatial features.

## Audit finding 2 — Unsafe temporal features in v1

The v1 pollutant lag and rolling columns were created from pollutant fields containing original observations, interpolation and proxy estimates.

Two-sided interpolation may depend on a future endpoint and cannot be reproduced safely during live forecasting. Proxy values may also represent another station rather than the local station history.

Therefore, all v2 pollutant lags and rolling statistics are rebuilt from `<pollutant>_observed` only. Observation-count features accompany rolling means so the model can distinguish well-supported windows from sparse windows.

In [38]:
df_v2["timestamp_hour"] = pd.to_datetime(
    df_v2["timestamp_hour"]
)

pollutant_lag_config = {
    "pm25": [1, 6, 12, 24],
    "pm10": [1, 6, 12, 24],
    "no2": [1, 6, 24],
    "co": [1, 6, 24],
    "o3": [1, 6, 24],
    "so2": [1, 6],
}

observed_lag_columns = []

for pollutant, lag_hours in pollutant_lag_config.items():
    observed_column = f"{pollutant}_observed"

    observed_lookup = df_v2.set_index(
        ["location_id", "timestamp_hour"]
    )[observed_column]

    for hour in lag_hours:
        past_index = pd.MultiIndex.from_arrays(
            [
                df_v2["location_id"],
                df_v2["timestamp_hour"] - pd.Timedelta(hours=hour),
            ],
            names=["location_id", "timestamp_hour"],
        )

        lag_column = f"{pollutant}_observed_lag_{hour}h"

        df_v2[lag_column] = (
            observed_lookup.reindex(past_index).to_numpy()
        )

        observed_lag_columns.append(lag_column)

In [39]:
print("Observed lag columns created:", len(observed_lag_columns))

print(
    df_v2[observed_lag_columns]
    .notna()
    .sum()
)

Observed lag columns created: 19
pm25_observed_lag_1h     62656
pm25_observed_lag_6h     61701
pm25_observed_lag_12h    61082
pm25_observed_lag_24h    60389
pm10_observed_lag_1h     62162
pm10_observed_lag_6h     61196
pm10_observed_lag_12h    60574
pm10_observed_lag_24h    59847
no2_observed_lag_1h      62748
no2_observed_lag_6h      61768
no2_observed_lag_24h     60242
co_observed_lag_1h       61619
co_observed_lag_6h       60658
co_observed_lag_24h      59127
o3_observed_lag_1h       61532
o3_observed_lag_6h       60570
o3_observed_lag_24h      59084
so2_observed_lag_1h      50290
so2_observed_lag_6h      49511
dtype: int64


In [40]:
import numpy as np 
rolling_config = {
    "pm25": [6, 12, 24],
    "pm10": [6, 12, 24],
    "no2": [6, 12],
    "co": [6, 12],
    "o3": [6, 12],
    "so2": [6],
}

observed_rolling_columns = []
observed_count_columns = []

for pollutant, windows in rolling_config.items():
    observed_column = f"{pollutant}_observed"

    for window in windows:
        mean_column = (
            f"{pollutant}_observed_rolling_mean_{window}h"
        )
        count_column = (
            f"{pollutant}_observed_count_{window}h"
        )

        df_v2[mean_column] = np.nan
        df_v2[count_column] = 0

        for _, station_rows in df_v2.groupby("location_id").groups.items():
            station_data = (
                df_v2.loc[
                    station_rows,
                    ["timestamp_hour", observed_column],
                ]
                .sort_values("timestamp_hour")
            )

            rolling = station_data.rolling(
                window=f"{window}h",
                on="timestamp_hour",
                min_periods=1,
            )[observed_column]

            rolling_mean = rolling.mean()
            rolling_count = rolling.count()

            minimum_required = max(2, window // 2)

            rolling_mean = rolling_mean.where(
                rolling_count >= minimum_required
            )

            df_v2.loc[
                station_data.index,
                mean_column,
            ] = rolling_mean.to_numpy()

            df_v2.loc[
                station_data.index,
                count_column,
            ] = rolling_count.to_numpy()

        observed_rolling_columns.append(mean_column)
        observed_count_columns.append(count_column)

In [41]:
print(
    "Rolling means created:",
    len(observed_rolling_columns)
)

print(
    "Rolling counts created:",
    len(observed_count_columns)
)

print(
    df_v2[observed_rolling_columns]
    .notna()
    .sum()
)

Rolling means created: 13
Rolling counts created: 13
pm25_observed_rolling_mean_6h     70262
pm25_observed_rolling_mean_12h    70952
pm25_observed_rolling_mean_24h    71194
pm10_observed_rolling_mean_6h     69431
pm10_observed_rolling_mean_12h    70122
pm10_observed_rolling_mean_24h    70242
no2_observed_rolling_mean_6h      70791
no2_observed_rolling_mean_12h     71336
co_observed_rolling_mean_6h       69892
co_observed_rolling_mean_12h      70531
o3_observed_rolling_mean_6h       69152
o3_observed_rolling_mean_12h      69857
so2_observed_rolling_mean_6h      57337
dtype: int64


## Exploratory neighbouring-station baseline

Neighbouring-station estimates are retained as separate spatial context features rather than being inserted into local pollutant histories.

Only same-timestamp original observations from other stations are eligible donors. The initial spatial baseline measures coverage, MAE and RMSE to determine whether neighbouring context contains useful signal.

Lower error indicates a better estimate for that pollutant and evaluation sample, but it does not prove reliability or causality. Coverage and distance must also be considered.

In [42]:
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    latitude_difference = lat2 - lat1
    longitude_difference = lon2 - lon1

    a = (
        sin(latitude_difference / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(longitude_difference / 2) ** 2
    )

    return (
        earth_radius_km
        * 2
        * atan2(sqrt(a), sqrt(1 - a))
    )

In [43]:
station_metadata = (
    df_v2[
        [
            "location_id",
            "station_name",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates("location_id")
    .reset_index(drop=True)
)

print("Stations:", len(station_metadata))

Stations: 38


In [44]:
MAX_NEIGHBOR_DISTANCE_KM = 15.0

distance_rows = []

for target in station_metadata.itertuples(index=False):
    for donor in station_metadata.itertuples(index=False):
        if target.location_id == donor.location_id:
            continue

        distance = haversine_km(
            target.latitude,
            target.longitude,
            donor.latitude,
            donor.longitude,
        )

        if distance <= MAX_NEIGHBOR_DISTANCE_KM:
            distance_rows.append(
                {
                    "target_location_id": target.location_id,
                    "target_station": target.station_name,
                    "donor_location_id": donor.location_id,
                    "donor_station": donor.station_name,
                    "distance_km": distance,
                }
            )

station_distances = (
    pd.DataFrame(distance_rows)
    .sort_values(
        ["target_location_id", "distance_km"]
    )
    .reset_index(drop=True)
)

In [45]:
print(station_distances.shape)

station_distances[
    station_distances["target_station"]
    == "Aya Nagar, New Delhi - IMD"
].head()

(712, 5)


,target_location_id,target_station,donor_location_id,donor_station,distance_km
83,5570,"Aya Nagar, New Delhi - IMD",10484,"Sri Aurobindo Marg, Delhi - DPCC",8.545492
84,5570,"Aya Nagar, New Delhi - IMD",5650,"IGI Airport (T3), Delhi - IMD",9.931749
85,5570,"Aya Nagar, New Delhi - IMD",17,"R K Puram, Delhi - DPCC",11.276727
86,5570,"Aya Nagar, New Delhi - IMD",5586,"Sirifort, Delhi - CPCB",11.816135
87,5570,"Aya Nagar, New Delhi - IMD",6931,"Dwarka-Sector 8, Delhi - DPCC",12.239358


In [46]:
MAX_NEIGHBORS = 3
IDW_POWER = 2

neighbor_feature_columns = []

for pollutant in pollutants:
    observed_column = f"{pollutant}_observed"
    estimate_column = f"{pollutant}_neighbor_idw"
    count_column = f"{pollutant}_neighbor_count"
    distance_column = f"{pollutant}_neighbor_nearest_km"

    donor_observations = (
        df_v2[
            [
                "location_id",
                "timestamp_hour",
                observed_column,
            ]
        ]
        .dropna(subset=[observed_column])
        .rename(
            columns={
                "location_id": "donor_location_id",
                observed_column: "donor_value",
            }
        )
    )

    candidates = station_distances[
        [
            "target_location_id",
            "donor_location_id",
            "distance_km",
        ]
    ].merge(
        donor_observations,
        on="donor_location_id",
        how="inner",
    )

    candidates = (
        candidates.sort_values(
            [
                "target_location_id",
                "timestamp_hour",
                "distance_km",
            ]
        )
        .groupby(
            ["target_location_id", "timestamp_hour"],
            as_index=False,
        )
        .head(MAX_NEIGHBORS)
    )

    candidates["weight"] = (
        1 / candidates["distance_km"] ** IDW_POWER
    )

    candidates["weighted_value"] = (
        candidates["donor_value"]
        * candidates["weight"]
    )

    neighbor_features = (
        candidates.groupby(
            ["target_location_id", "timestamp_hour"],
            as_index=False,
        )
        .agg(
            weighted_value_sum=("weighted_value", "sum"),
            weight_sum=("weight", "sum"),
            neighbor_count=("donor_value", "count"),
            nearest_distance_km=("distance_km", "min"),
        )
    )

    neighbor_features[estimate_column] = (
        neighbor_features["weighted_value_sum"]
        / neighbor_features["weight_sum"]
    )

    feature_lookup = neighbor_features.set_index(
        ["target_location_id", "timestamp_hour"]
    )

    target_index = pd.MultiIndex.from_arrays(
        [
            df_v2["location_id"],
            df_v2["timestamp_hour"],
        ],
        names=["target_location_id", "timestamp_hour"],
    )

    df_v2[estimate_column] = (
        feature_lookup[estimate_column]
        .reindex(target_index)
        .to_numpy()
    )

    df_v2[count_column] = (
        feature_lookup["neighbor_count"]
        .reindex(target_index)
        .fillna(0)
        .to_numpy()
    )

    df_v2[distance_column] = (
        feature_lookup["nearest_distance_km"]
        .reindex(target_index)
        .to_numpy()
    )

    neighbor_feature_columns.extend(
        [
            estimate_column,
            count_column,
            distance_column,
        ]
    )

In [47]:
for pollutant in pollutants:
    observed = df_v2[f"{pollutant}_observed"]
    estimate = df_v2[f"{pollutant}_neighbor_idw"]

    comparable = observed.notna() & estimate.notna()

    errors = (
        observed[comparable]
        - estimate[comparable]
    )

    mae = errors.abs().mean()
    rmse = np.sqrt((errors ** 2).mean())

    print(
        pollutant,
        "coverage:",
        round(estimate.notna().mean() * 100, 2),
        "comparable rows:",
        comparable.sum(),
        "MAE:",
        round(mae, 2),
        "RMSE:",
        round(rmse, 2),
    )

pm25 coverage: 91.21 comparable rows: 62812 MAE: 17.72 RMSE: 30.5
pm10 coverage: 91.21 comparable rows: 62318 MAE: 45.32 RMSE: 72.78
no2 coverage: 91.21 comparable rows: 62897 MAE: 18.32 RMSE: 25.67
co coverage: 91.2 comparable rows: 61772 MAE: 0.46 RMSE: 0.71
so2 coverage: 90.92 comparable rows: 50424 MAE: 9.51 RMSE: 17.37
o3 coverage: 91.18 comparable rows: 61690 MAE: 16.1 RMSE: 24.43


## Chronological selection of pollutant-specific radii

Candidate radii are compared using a chronological validation period rather than random sampling.

A separate radius is selected for each pollutant by balancing coverage with MAE and RMSE. The final chronological test period is used only once after radius selection.

The selected radii are:

- PM10 and CO: 12.5 km
- PM2.5, NO2, SO2 and O3: 15 km

Up to three nearest available donors are combined using inverse-distance weighting with power 2.

In [48]:
# ---------------------------------------------------------
# Tune the neighbour radius on a chronological validation set
# ---------------------------------------------------------

df_v2["timestamp_hour"] = pd.to_datetime(df_v2["timestamp_hour"])

# 60% development | 20% validation | 20% final test
unique_times = np.sort(df_v2["timestamp_hour"].unique())

validation_start = pd.Timestamp(unique_times[int(len(unique_times) * 0.60)])
test_start = pd.Timestamp(unique_times[int(len(unique_times) * 0.80)])

validation_mask = (
    (df_v2["timestamp_hour"] >= validation_start)
    & (df_v2["timestamp_hour"] < test_start)
)

print("Validation:", validation_start, "to", test_start)
print("Final test starts:", test_start)


def evaluate_neighbor_radius(
    pollutant,
    radius_km,
    validation_mask,
    max_neighbors=3,
    idw_power=2
):
    observed_column = f"{pollutant}_observed"

    # Only observed donor measurements from the validation period
    donors = (
        df_v2.loc[
            validation_mask & df_v2[observed_column].notna(),
            ["location_id", "timestamp_hour", observed_column]
        ]
        .rename(columns={
            "location_id": "donor_location_id",
            observed_column: "donor_value"
        })
    )

    # Keep station pairs inside the radius
    eligible_distances = station_distances[
        station_distances["distance_km"] <= radius_km
    ]

    # Match every target station with available donors at the same time
    candidates = eligible_distances.merge(
        donors,
        on="donor_location_id",
        how="inner"
    )

    if candidates.empty:
        return {
            "pollutant": pollutant,
            "radius_km": radius_km,
            "coverage": 0,
            "comparable_rows": 0,
            "mae": np.nan,
            "rmse": np.nan
        }

    # Select the nearest available donors
    candidates = (
        candidates
        .sort_values(
            ["target_location_id", "timestamp_hour", "distance_km"]
        )
        .groupby(
            ["target_location_id", "timestamp_hour"],
            sort=False
        )
        .head(max_neighbors)
        .copy()
    )

    # Inverse-distance weighting
    candidates["weight"] = (
        1 / candidates["distance_km"].pow(idw_power)
    )

    candidates["weighted_value"] = (
        candidates["donor_value"] * candidates["weight"]
    )

    estimates = (
        candidates
        .groupby(
            ["target_location_id", "timestamp_hour"],
            as_index=False
        )
        .agg(
            weighted_sum=("weighted_value", "sum"),
            weight_sum=("weight", "sum"),
            neighbor_count=("donor_location_id", "size"),
            nearest_km=("distance_km", "min")
        )
    )

    estimates["neighbor_estimate"] = (
        estimates["weighted_sum"] / estimates["weight_sum"]
    )

    # Actual local observations used only for evaluation
    targets = (
        df_v2.loc[
            validation_mask,
            ["location_id", "timestamp_hour", observed_column]
        ]
        .rename(columns={
            "location_id": "target_location_id",
            observed_column: "actual_value"
        })
    )

    scored = targets.merge(
        estimates[
            [
                "target_location_id",
                "timestamp_hour",
                "neighbor_estimate"
            ]
        ],
        on=["target_location_id", "timestamp_hour"],
        how="left"
    )

    coverage = scored["neighbor_estimate"].notna().mean() * 100

    comparable = scored.dropna(
        subset=["actual_value", "neighbor_estimate"]
    )

    errors = (
        comparable["neighbor_estimate"]
        - comparable["actual_value"]
    )

    return {
        "pollutant": pollutant,
        "radius_km": radius_km,
        "coverage": coverage,
        "comparable_rows": len(comparable),
        "mae": errors.abs().mean(),
        "rmse": np.sqrt((errors ** 2).mean())
    }


# Test these distance limits
radius_options = [5, 7.5, 10, 12.5, 15]

radius_results = []

for pollutant in pollutants:
    print("Testing:", pollutant)

    for radius in radius_options:
        result = evaluate_neighbor_radius(
            pollutant=pollutant,
            radius_km=radius,
            validation_mask=validation_mask
        )

        radius_results.append(result)

radius_results_df = pd.DataFrame(radius_results)

display(
    radius_results_df.round(2)
    .sort_values(["pollutant", "radius_km"])
)

Validation: 2026-06-02 23:00:00 to 2026-06-20 03:00:00
Final test starts: 2026-06-20 03:00:00
Testing: pm25
Testing: pm10
Testing: no2
Testing: co
Testing: so2
Testing: o3


,pollutant,radius_km,coverage,comparable_rows,mae,rmse
15,co,5.0,68.27,9280,0.45,0.60
16,co,7.5,82.68,11236,0.42,0.57
17,co,10.0,86.04,11608,0.42,0.57
18,co,12.5,87.97,11874,0.42,0.57
19,co,15.0,88.57,11958,0.42,0.58
10,no2,5.0,68.76,9586,17.46,23.99
11,no2,7.5,82.76,11540,17.60,24.79
12,no2,10.0,86.05,11913,17.45,24.44
13,no2,12.5,87.97,12177,17.57,24.55
14,no2,15.0,88.58,12260,17.57,24.55


In [49]:
# Choose the lowest-RMSE radius whose coverage is
# within 2 percentage points of that pollutant's best coverage

best_radius_rows = []

for pollutant, results in radius_results_df.groupby("pollutant"):
    maximum_coverage = results["coverage"].max()

    acceptable_results = results[
        results["coverage"] >= maximum_coverage - 2
    ]

    best_result = (
        acceptable_results
        .sort_values(["rmse", "mae", "radius_km"])
        .iloc[0]
    )

    best_radius_rows.append(best_result)

best_radii = pd.DataFrame(best_radius_rows).reset_index(drop=True)

display(best_radii.round(2))

,pollutant,radius_km,coverage,comparable_rows,mae,rmse
0,co,12.5,87.97,11874,0.42,0.57
1,no2,15.0,88.58,12260,17.57,24.55
2,o3,15.0,88.56,12036,16.00,23.73
3,pm10,12.5,87.97,12078,43.18,71.47
4,pm25,15.0,88.57,12231,15.75,26.46
5,so2,15.0,88.39,9819,8.59,11.64


In [ ]:
import gc

df_v2 = df_v2.copy()

selected_radii = dict(
    zip(best_radii["pollutant"], best_radii["radius_km"])
)

print("Selected radii:", selected_radii)


def build_final_neighbor_features(
    pollutant,
    radius_km,
    max_neighbors=3,
    idw_power=2
):
    observed_column = f"{pollutant}_observed"

    donors = (
        df_v2.loc[
            df_v2[observed_column].notna(),
            ["location_id", "timestamp_hour", observed_column]
        ]
        .rename(columns={
            "location_id": "donor_location_id",
            observed_column: "donor_value"
        })
    )

    eligible_distances = station_distances.loc[
        station_distances["distance_km"] <= radius_km,
        [
            "target_location_id",
            "donor_location_id",
            "distance_km"
        ]
    ]

    candidates = eligible_distances.merge(
        donors,
        on="donor_location_id",
        how="inner"
    )

    # Keep the three nearest available donors
    candidates = (
        candidates
        .sort_values(
            ["target_location_id", "timestamp_hour", "distance_km"]
        )
        .groupby(
            ["target_location_id", "timestamp_hour"],
            sort=False
        )
        .head(max_neighbors)
        .copy()
    )

    candidates["weight"] = (
        1 / candidates["distance_km"].pow(idw_power)
    )

    candidates["weighted_value"] = (
        candidates["donor_value"] * candidates["weight"]
    )

    estimates = (
        candidates
        .groupby(
            ["target_location_id", "timestamp_hour"],
            as_index=False
        )
        .agg(
            weighted_sum=("weighted_value", "sum"),
            weight_sum=("weight", "sum"),
            neighbor_count=("donor_location_id", "size"),
            nearest_km=("distance_km", "min")
        )
    )

    estimates["neighbor_estimate"] = (
        estimates["weighted_sum"] / estimates["weight_sum"]
    )

    estimate_lookup = estimates.set_index(
        ["target_location_id", "timestamp_hour"]
    )

    row_index = pd.MultiIndex.from_arrays(
        [
            df_v2["location_id"],
            df_v2["timestamp_hour"]
        ],
        names=["target_location_id", "timestamp_hour"]
    )

    prefix = f"{pollutant}_neighbor"

    df_v2[f"{prefix}_idw_v2"] = (
        estimate_lookup["neighbor_estimate"]
        .reindex(row_index)
        .to_numpy()
    )

    df_v2[f"{prefix}_count_v2"] = (
        estimate_lookup["neighbor_count"]
        .reindex(row_index)
        .fillna(0)
        .astype("int8")
        .to_numpy()
    )

    df_v2[f"{prefix}_nearest_km_v2"] = (
        estimate_lookup["nearest_km"]
        .reindex(row_index)
        .to_numpy()
    )

    del candidates, estimates, estimate_lookup
    gc.collect()


for pollutant in pollutants:
    radius = selected_radii[pollutant]

    print(f"Building {pollutant}: radius={radius} km")

    build_final_neighbor_features(
        pollutant=pollutant,
        radius_km=radius
    )

print("Final spatial features created.")

In [51]:
test_mask = df_v2["timestamp_hour"] >= test_start

test_results = []

for pollutant in pollutants:
    actual_column = f"{pollutant}_observed"
    estimate_column = f"{pollutant}_neighbor_idw_v2"

    test_data = df_v2.loc[
        test_mask,
        [actual_column, estimate_column]
    ]

    coverage = (
        test_data[estimate_column].notna().mean() * 100
    )

    comparable = test_data.dropna(
        subset=[actual_column, estimate_column]
    )

    errors = (
        comparable[estimate_column]
        - comparable[actual_column]
    )

    test_results.append({
        "pollutant": pollutant,
        "radius_km": selected_radii[pollutant],
        "coverage": coverage,
        "comparable_rows": len(comparable),
        "mae": errors.abs().mean(),
        "rmse": np.sqrt((errors ** 2).mean())
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df.round(2))

,pollutant,radius_km,coverage,comparable_rows,mae,rmse
0,pm25,15.0,93.65,12756,15.76,26.35
1,pm10,12.5,93.24,12985,34.91,55.25
2,no2,15.0,93.67,13250,15.23,21.25
3,co,12.5,93.24,13238,0.43,0.63
4,so2,15.0,93.63,10983,9.68,15.10
5,o3,15.0,93.67,13013,16.55,25.68


In [53]:
# Ensure chronological order within each station
df_v2 = (
    df_v2
    .sort_values(["location_id", "timestamp_hour"])
    .reset_index(drop=True)
)

for pollutant in pollutants:
    has_column = f"has_{pollutant}"

    # Uses only the current and previous timestamps
    seen_so_far = (
        df_v2
        .groupby("location_id")[has_column]
        .cummax()
        .astype(bool)
    )

    df_v2[f"station_seen_{pollutant}_so_far"] = seen_so_far

    df_v2[f"{pollutant}_temporary_missing_causal"] = (
        seen_so_far & ~df_v2[has_column]
    )

    df_v2[f"{pollutant}_not_yet_observed_causal"] = (
        ~seen_so_far
    )

print("Causal station-history flags created.")

Causal station-history flags created.


In [54]:
for pollutant in pollutants:
    print(
        pollutant,
        "seen so far:",
        df_v2[f"station_seen_{pollutant}_so_far"].sum(),
        "temporary missing:",
        df_v2[f"{pollutant}_temporary_missing_causal"].sum(),
        "not yet observed:",
        df_v2[f"{pollutant}_not_yet_observed_causal"].sum()
    )

pm25 seen so far: 76463 temporary missing: 13635 not yet observed: 1741
pm10 seen so far: 74387 temporary missing: 12053 not yet observed: 3817
no2 seen so far: 76466 temporary missing: 13547 not yet observed: 1738
co seen so far: 76466 temporary missing: 14680 not yet observed: 1738
so2 seen so far: 62053 temporary missing: 11603 not yet observed: 16151
o3 seen so far: 74408 temporary missing: 12704 not yet observed: 3796


In [55]:
# ---------------------------------------------------------
# Define the safe model feature set
# ---------------------------------------------------------

aqi_context_columns = [
    "current_aqi",
    "aqi_calculation_valid",
    "aqi_lag_1h",
    "aqi_lag_6h",
    "aqi_lag_12h",
    "aqi_lag_24h"
]

geographic_columns = [
    "latitude",
    "longitude"
]

time_columns = [
    "hour",
    "day_of_week",
    "is_weekend",
    "month",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos"
]

weather_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "is_raining",
    "wind_speed_10m_ms",
    "wind_u_ms",
    "wind_v_ms"
]

observation_columns = [
    f"{pollutant}_observed"
    for pollutant in pollutants
]

availability_columns = (
    [f"has_{pollutant}" for pollutant in pollutants]
    + [
        f"station_seen_{pollutant}_so_far"
        for pollutant in pollutants
    ]
    + [
        f"{pollutant}_temporary_missing_causal"
        for pollutant in pollutants
    ]
    + [
        f"{pollutant}_not_yet_observed_causal"
        for pollutant in pollutants
    ]
)

observed_lag_columns = [
    column for column in df_v2.columns
    if "_observed_lag_" in column
]

observed_rolling_columns = [
    column for column in df_v2.columns
    if "_observed_rolling_mean_" in column
]

rolling_count_columns = [
    column for column in df_v2.columns
    if "_observed_count_" in column
]

spatial_columns = [
    column for column in df_v2.columns
    if (
        "_neighbor_idw_v2" in column
        or "_neighbor_count_v2" in column
        or "_neighbor_nearest_km_v2" in column
    )
]

model_feature_columns = list(dict.fromkeys(
    geographic_columns
    + aqi_context_columns
    + time_columns
    + weather_columns
    + observation_columns
    + availability_columns
    + observed_lag_columns
    + observed_rolling_columns
    + rolling_count_columns
    + spatial_columns
))

target_columns = [
    f"target_aqi_{hour}h"
    for hour in range(1, 25)
]

In [56]:
# ---------------------------------------------------------
# Final leakage and integrity assertions
# ---------------------------------------------------------

# 1. Unique station-hour rows
duplicate_count = df_v2.duplicated(
    ["location_id", "timestamp_hour"]
).sum()

assert duplicate_count == 0, "Duplicate station-hour rows found."


# 2. Every requested model feature exists
missing_features = [
    column for column in model_feature_columns
    if column not in df_v2.columns
]

assert not missing_features, f"Missing features: {missing_features}"


# 3. Forbidden legacy columns must not enter the model
forbidden_columns = (
    pollutants
    + [f"{pollutant}_interpolated" for pollutant in pollutants]
    + [f"{pollutant}_proxy_imputed" for pollutant in pollutants]
    + [f"station_observed_{pollutant}" for pollutant in pollutants]
    + [f"{pollutant}_temporary_missing" for pollutant in pollutants]
    + [f"{pollutant}_structural_missing" for pollutant in pollutants]
    + [f"{pollutant}_neighbor_estimate_v1" for pollutant in pollutants]
    + ["station_reliability"]
)

legacy_overlap = sorted(
    set(model_feature_columns) & set(forbidden_columns)
)

assert not legacy_overlap, (
    f"Legacy features entered model set: {legacy_overlap}"
)


# 4. Observed values must exactly match original availability
for pollutant in pollutants:
    observed_available = (
        df_v2[f"{pollutant}_observed"].notna()
    )

    original_available = (
        df_v2[f"has_{pollutant}"].astype(bool)
    )

    assert observed_available.equals(original_available), (
        f"{pollutant}: observed/availability mismatch"
    )


# 5. Validate causal station-history flags
for pollutant in pollutants:
    has_value = df_v2[f"has_{pollutant}"].astype(bool)
    seen = df_v2[f"station_seen_{pollutant}_so_far"]
    temporary = df_v2[f"{pollutant}_temporary_missing_causal"]
    not_yet = df_v2[f"{pollutant}_not_yet_observed_causal"]

    assert seen.equals(~not_yet), (
        f"{pollutant}: causal flags are not complementary"
    )

    assert temporary.equals(seen & ~has_value), (
        f"{pollutant}: temporary-missing flag is incorrect"
    )


# 6. Spatial distances must respect selected radii
for pollutant in pollutants:
    distance_column = (
        f"{pollutant}_neighbor_nearest_km_v2"
    )

    distances = df_v2[distance_column].dropna()

    assert (distances > 0).all(), (
        f"{pollutant}: self-neighbour detected"
    )

    assert (
        distances <= selected_radii[pollutant] + 1e-9
    ).all(), (
        f"{pollutant}: neighbour outside selected radius"
    )


# 7. Model features must not contain infinity
numeric_features = (
    df_v2[model_feature_columns]
    .select_dtypes(include=np.number)
)

infinite_count = np.isinf(numeric_features).sum().sum()

assert infinite_count == 0, "Infinite feature values detected."


# 8. Targets must never be model predictors
target_overlap = (
    set(model_feature_columns) & set(target_columns)
)

assert not target_overlap, "Target leakage detected."


print("FINAL INTEGRITY CHECK: PASSED")
print("Rows:", len(df_v2))
print("Model features:", len(model_feature_columns))
print("Targets:", len(target_columns))
print("Spatial features:", len(spatial_columns))
print("Duplicate station-hours:", duplicate_count)
print("Infinite values:", infinite_count)

FINAL INTEGRITY CHECK: PASSED
Rows: 78204
Model features: 117
Targets: 24
Spatial features: 18
Duplicate station-hours: 0
Infinite values: 0


In [57]:
from pathlib import Path
import json

# ---------------------------------------------------------
# Create the final v2 dataset
# ---------------------------------------------------------

identifier_columns = [
    "location_id",
    "station_name",
    "provider_name",
    "timestamp_hour"
]

export_columns = list(dict.fromkeys(
    identifier_columns
    + model_feature_columns
    + target_columns
))

features_core_v2 = (
    df_v2[export_columns]
    .sort_values(["location_id", "timestamp_hour"])
    .reset_index(drop=True)
)

print("Export shape:", features_core_v2.shape)

Export shape: (78204, 145)


In [ ]:
# ---------------------------------------------------------
# Locate project and save files
# ---------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root != project_root.parent
    and not (project_root / "data").exists()
):
    project_root = project_root.parent

if not (project_root / "data").exists():
    raise FileNotFoundError(
        "Could not locate the ClearTrace data directory."
    )

output_directory = project_root / "data" / "features"
output_directory.mkdir(parents=True, exist_ok=True)

csv_path = output_directory / "features_core_v2.csv"
manifest_path = output_directory / "features_core_v2_manifest.json"

features_core_v2.to_csv(csv_path, index=False)

manifest = {
    "dataset_version": "features_core_v2",
    "rows": len(features_core_v2),
    "columns": len(features_core_v2.columns),
    "model_feature_count": len(model_feature_columns),
    "target_count": len(target_columns),
    "model_features": model_feature_columns,
    "targets": target_columns,
    "spatial_configuration": {
        "selected_radius_km": {
            pollutant: float(radius)
            for pollutant, radius in selected_radii.items()
        },
        "maximum_neighbors": 3,
        "idw_power": 2,
        "same_timestamp_donors": True,
        "observed_donors_only": True
    },
    "temporal_split": {
        "validation_start": str(validation_start),
        "test_start": str(test_start)
    },
    "excluded_legacy_features": [
        "mixed pollutant columns",
        "bidirectional interpolation features",
        "legacy proxy-imputed values",
        "legacy pollutant lag features",
        "legacy pollutant rolling features",
        "future-aware station availability flags",
        "full-period station reliability"
    ]
}

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print("Saved CSV:", csv_path)
print("Saved manifest:", manifest_path)
print(
    "CSV size (MB):",
    round(csv_path.stat().st_size / 1_000_000, 2)
)

In [59]:
saved_v2 = pd.read_csv(
    csv_path,
    parse_dates=["timestamp_hour"]
)

assert saved_v2.shape == features_core_v2.shape
assert saved_v2.columns.tolist() == export_columns

assert saved_v2.duplicated(
    ["location_id", "timestamp_hour"]
).sum() == 0

print("SAVED V2 VERIFICATION: PASSED")
print("Shape:", saved_v2.shape)

SAVED V2 VERIFICATION: PASSED
Shape: (78204, 145)
